# Iceberg Basics — 06: Schema Evolution


Iceberg has the richest schema evolution support of any table format.
Column operations are metadata-only — no data rewrite required.

Topics: ADD/DROP/RENAME/REORDER columns, type promotion, column IDs vs names.


In [1]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:38:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Iceberg catalog ready


26/04/10 20:38:17 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:38:19 WARN TaskSetManager: Stage 1 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


## Step 1 — Create Table and Add Columns

ADD COLUMN is always metadata-only in Iceberg.

In [2]:

spark.sql("DROP TABLE IF EXISTS local.icedb.schema_orders")
df.writeTo("local.icedb.schema_orders").using("iceberg").createOrReplace()
print("Original schema:")
spark.table("local.icedb.schema_orders").printSchema()

spark.sql("""
    ALTER TABLE local.icedb.schema_orders
    ADD COLUMNS (
        discount_pct  DOUBLE  COMMENT 'Discount applied 0.0-1.0',
        loyalty_pts   INT     COMMENT 'Loyalty points earned',
        channel       STRING  COMMENT 'Sales channel: web/mobile/store'
    )
""")
print("\nAfter ADD COLUMNS (3 new columns, NO data rewrite):")
spark.table("local.icedb.schema_orders").printSchema()
print()
print("Old rows have null for new columns:")
spark.table("local.icedb.schema_orders").select("order_id","revenue","discount_pct","channel").show(5)


26/04/10 20:38:20 WARN TaskSetManager: Stage 4 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:38:22 WARN HadoopTableOperations: Error reading version hint file /workspace/data/warehouse/iceberg/icedb/schema_orders/metadata/version-hint.text
java.io.FileNotFoundException: File /workspace/data/warehouse/iceberg/icedb/schema_orders/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:189)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:572)
	at org.apache.hadoo

Original schema:
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)


After ADD COLUMNS (3 new columns, NO data rewrite):
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- loyalty_pts: integer (nullable = true)
 |-- channel: string (nullable = true)


Old ro

## Step 2 — RENAME and DROP Columns

Rename and drop are also metadata-only.

In [3]:

spark.sql("ALTER TABLE local.icedb.schema_orders RENAME COLUMN channel TO sales_channel")
print("Renamed: channel → sales_channel")

spark.sql("ALTER TABLE local.icedb.schema_orders DROP COLUMN loyalty_pts")
print("Dropped: loyalty_pts")
spark.table("local.icedb.schema_orders").printSchema()


Renamed: channel → sales_channel
Dropped: loyalty_pts
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- sales_channel: string (nullable = true)



## Step 3 — Type Promotion

Safe widening promotions: int→long, float→double.

In [4]:

TYPE_TABLE = "local.icedb.type_test"
spark.sql(f"DROP TABLE IF EXISTS {TYPE_TABLE}")
spark.createDataFrame([(i, float(i), f"name_{i}") for i in range(1000)],
    ["id","value","name"]).writeTo(TYPE_TABLE).using("iceberg").createOrReplace()

spark.sql(f"ALTER TABLE {TYPE_TABLE} ALTER COLUMN id TYPE BIGINT")
print("INT → BIGINT: safe widening ✅")
spark.sql(f"ALTER TABLE {TYPE_TABLE} ALTER COLUMN value TYPE DOUBLE")
print("FLOAT → DOUBLE: safe widening ✅")
spark.table(TYPE_TABLE).printSchema()
print(f"All {spark.table(TYPE_TABLE).count()} rows readable after type change")


26/04/10 20:38:24 WARN HadoopTableOperations: Error reading version hint file /workspace/data/warehouse/iceberg/icedb/type_test/metadata/version-hint.text
java.io.FileNotFoundException: File /workspace/data/warehouse/iceberg/icedb/type_test/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:189)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:572)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:997)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apa

INT → BIGINT: safe widening ✅
FLOAT → DOUBLE: safe widening ✅
root
 |-- id: long (nullable = true)
 |-- value: double (nullable = true)
 |-- name: string (nullable = true)

All 1000 rows readable after type change


## Step 4 — Iceberg Column IDs (vs Names)

Iceberg uses stable numeric IDs, not names, to track columns.

In [5]:

# Show column IDs from Iceberg metadata
import json as pyjson, pathlib
warehouse = "/workspace/data/warehouse/iceberg"
meta_dir = pathlib.Path(f"{warehouse}/icedb/schema_orders/metadata")
meta_files = sorted(meta_dir.glob("*.json"))
if meta_files:
    meta = pyjson.loads(meta_files[-1].read_text())
    schema = meta.get("current-schema", meta.get("schema", {}))
    print("Iceberg column IDs (stable identifiers):")
    for field in schema.get("fields", []):
        print(f"  id={field['id']:<4} name={field['name']:<20} type={field['type']}")
    print()
    print("When you RENAME a column, the id stays the same.")
    print("Files written before the rename are still readable — no data rewrite.")
    print("This is why Iceberg rename is truly safe — file readers use IDs, not names.")


Iceberg column IDs (stable identifiers):

When you RENAME a column, the id stays the same.
Files written before the rename are still readable — no data rewrite.
This is why Iceberg rename is truly safe — file readers use IDs, not names.


## Summary



In [6]:

print("""
ALTER TABLE local.db.table ADD COLUMNS (new_col STRING)
ALTER TABLE local.db.table RENAME COLUMN old TO new
ALTER TABLE local.db.table DROP COLUMN col_name
ALTER TABLE local.db.table ALTER COLUMN id TYPE BIGINT

Safe type promotions: INT→LONG, FLOAT→DOUBLE, DECIMAL(p,s)→DECIMAL(p2,s) if p2>p
Iceberg uses column IDs internally → rename is truly metadata-only
Old data files automatically readable with new schema (IDs match)
""")



ALTER TABLE local.db.table ADD COLUMNS (new_col STRING)
ALTER TABLE local.db.table RENAME COLUMN old TO new
ALTER TABLE local.db.table DROP COLUMN col_name
ALTER TABLE local.db.table ALTER COLUMN id TYPE BIGINT

Safe type promotions: INT→LONG, FLOAT→DOUBLE, DECIMAL(p,s)→DECIMAL(p2,s) if p2>p
Iceberg uses column IDs internally → rename is truly metadata-only
Old data files automatically readable with new schema (IDs match)

